> **Modelo D5 (no el final).** Este notebook se ejecutó con `d5_ema_w32_e60_s42.pth`, el candidato de la primera evaluación en test y sin orientación canónica. Sirvió para encontrar el problema de orientación que llevó a H6 (`docs/decisiones_semana10.md`, secciones 12.4 a 12.6). Las cifras del modelo final H6 están en `notebooks/04_semana10_pipeline_multitarea.ipynb` y en `docs/resumen_semana10.md`.

# 05 · Casos difíciles: mirar, medir y probar qué pasa si se reorientan

**Para qué sirve.** En la evaluación de test el Dice de instancias quedó en 0.845 (objetivo 0.85).
Trece de los quince casos de test están entre 0.825 y 0.926; los casos **012** (0.692) y **039**
(0.710) arrastran el promedio. Este notebook permite verlos corte a corte junto a los dos peores
de validación (**087** y **078**) y probar reorientaciones.

**Regla.** Los casos 012 y 039 son de test: aquí se abren solo para entender el fallo. Las
decisiones (qué modelo, qué preprocesamiento) se toman con validación; ver
`docs/decisiones_semana10.md`, sección 12.

**Qué necesita.** PyTorch, el checkpoint y los datos. Corre en dos sitios sin tocar nada:

- **Colab** (recomendado, con GPU): usa el código de `/content/codigo` y el caché de `/content/datos`.
  Cada caso tarda unos segundos.
- **Local**: usa `outputs/s10/` y `data/raw/`; la primera vez construye el caché de cada caso.
  En CPU cada caso tarda de 1 a 4 minutos.

Colores en todas las figuras: **azul** sacro, **rojo** coxal izquierdo, **verde** coxal derecho;
los fragmentos secundarios salen en otros tonos.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
import torch

# Funciona en el repo local y en Colab (código en /content/codigo, caché en /content/datos).
candidatos = [Path.cwd(), *Path.cwd().parents, Path("/content/codigo")]
ROOT = next(p for p in candidatos if (p / "src" / "casos_dificiles.py").exists())
os.chdir(ROOT)                      # las rutas de data/raw son relativas a la raíz del repo
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import casos_dificiles as cd
from orientacion import LPI_CASES, to_canonical

CACHE_ROOT = next((p for p in (Path("/content/datos"), ROOT / "data/cache/s10") if (p / "cache_384").exists()),
                  ROOT / "data/cache/s10")          # si no existe, se construye desde data/raw
CHECKPOINT = next(p for p in (ROOT / "outputs/s10/d5_ema_w32_e60_s42.pth",
                              Path("/content/d5/d5_ema_w32_e60_s42.pth")) if p.exists())   # modelo congelado de la semana 10
CASOS = ["012", "039", "087", "078"]                 # 012 y 039: test (solo diagnóstico); 087 y 078: validación
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

modelo, device = cd.cargar_modelo(CHECKPOINT, DEVICE, tta=True)
print("raíz:", ROOT, "| caché:", CACHE_ROOT, "| modelo:", CHECKPOINT, "| dispositivo:", DEVICE)

## 1. Cómo viene guardado cada archivo

La cabecera del `.mha` trae la matriz de dirección. En **34 de los 100** casos es
`-1 0 0 0 -1 0 0 0 1` (orientación LPI): el volumen está guardado girado 180° en el plano axial
respecto a los otros 66 (RAI). El lector actual (`SimpleITK.GetArrayFromImage`) no aplica esa
matriz, así que esos casos entran al modelo con el sacro arriba y el coxal izquierdo a la
izquierda de la imagen, al revés que el resto.

In [ ]:
cabeceras = pd.DataFrame([cd.leer_cabecera(c) for c in CASOS]).set_index("caso")
display(cabeceras)

todos = sorted(Path("data/raw").glob("PENGWIN_CT_train_images_part*/*.mha"))
if todos:
    girados = [p.stem for p in todos if cd.leer_cabecera(p.stem)["guardado_girado_180"]]
    print(f"{len(girados)} de {len(todos)} casos están guardados girados 180° (LPI)")
    print("coincide con la lista de src/orientacion.py:", set(girados) == set(LPI_CASES) if len(todos) == 100 else "(faltan archivos)")

## 2. Predicción y métricas por caso

- `hueso_tomado_por_otra_region_%`: hueso real que el modelo asigna a otra región (por ejemplo
  coxal izquierdo marcado como derecho). En validación es prácticamente 0.
- `hueso_predicho_como_fondo_%`: hueso real que el modelo no ve.

In [ ]:
datos, pred, met = {}, {}, {}
for c in CASOS:
    datos[c] = cd.cargar_caso(c, CACHE_ROOT)
    pred[c] = cd.predecir(modelo, datos[c]["images"], device)
    met[c] = cd.metricas(pred[c], datos[c]["labels"])
    print(c, met[c]["resumen"])

resumen = pd.DataFrame({c: met[c]["resumen"] for c in CASOS}).T
resumen.insert(0, "orientacion", [cabeceras.loc[c, "orientacion"] for c in CASOS])
resumen

## 3. Ver los cortes

Primero un mosaico de cuatro cortes de cada caso, repartidos a lo largo del hueso; después un
deslizador para recorrer el volumen completo (solo funciona con el notebook en ejecución).

In [ ]:
CASO = "012"
for c in CASOS:                     # cuatro cortes de cada caso; sube n para ver más
    cd.mosaico(datos[c]["images"], datos[c]["labels"], pred[c], f"caso {c}", n=4)
    plt.show()

In [ ]:
# Deslizador para recorrer el volumen (necesita ipywidgets y el notebook abierto en Jupyter/Colab/VS Code).
# En ejecución por lotes se salta con la variable de entorno SIN_DESLIZADOR=1.
if os.environ.get("SIN_DESLIZADOR"):
    print("deslizador omitido en la ejecución por lotes")
else:
    try:
        from ipywidgets import interact, IntSlider, Dropdown

        def ver(caso, z):
            cd.mostrar(datos[caso]["images"], datos[caso]["labels"], pred[caso], z, f"caso {caso}")
            plt.show()

        interact(ver, caso=Dropdown(options=CASOS, value=CASO),
                 z=IntSlider(min=0, max=max(len(datos[c]["images"]) for c in CASOS) - 1, value=120, continuous_update=False))
    except ImportError:
        print("pip install ipywidgets para el deslizador; mientras tanto usa la celda siguiente")

In [ ]:
# Un corte concreto. Los peores cortes de cada caso:
for c in CASOS:
    print(c, "peores cortes (z, Dice):", [(z, round(d, 2)) for z, d in sorted(met[c]["por_corte"], key=lambda p: p[1])[:6]])

cd.mostrar(datos["039"]["images"], datos["039"]["labels"], pred["039"], 147, "caso 039")
plt.show()

## 4. Dice de instancias a lo largo del volumen

La línea punteada es el objetivo (0.85). Sirve para ver si un caso falla en todo el volumen o
solo en una zona (por ejemplo el acetábulo, donde están las fracturas conminutas).

In [ ]:
cd.curva_por_corte({f"caso {c}": met[c] for c in CASOS})
plt.show()

## 5. ¿Y si se reorienta antes de entrar al modelo?

### 5a. Los casos guardados girados 180° (por cabecera)

Se gira el caso a la orientación común y se vuelve a predecir **con el mismo modelo**. Este
modelo se entrenó viendo las dos orientaciones mezcladas, así que no se espera un salto aquí:
la ganancia de la orientación canónica viene de **reentrenar** con todos los casos orientados
igual (experimento H4), porque entonces izquierda y derecha dejan de depender de adivinar la
orientación del corte.

In [ ]:
filas = []
for c in CASOS:
    if c not in LPI_CASES:
        continue
    im, lb = to_canonical(datos[c]["images"], c), to_canonical(datos[c]["labels"], c)
    p = cd.predecir(modelo, im, device)
    m = cd.metricas(p, lb)["resumen"]
    filas.append({"caso": c, "tal como viene": met[c]["resumen"]["dice_instancias"], "girado 180° (canónico)": m["dice_instancias"]})
    z = sorted(met[c]["por_corte"], key=lambda q: q[1])[len(met[c]["por_corte"]) // 2][0]
    cd.mostrar(datos[c]["images"], datos[c]["labels"], pred[c], z, f"caso {c} tal como viene,")
    cd.mostrar(im, lb, p, z, f"caso {c} en orientación canónica,")
    plt.show()
pd.DataFrame(filas)

### 5b. Un paciente girado de verdad (caso 012)

La cabecera del 012 es la normal (RAI): aquí no hay metadato que corregir, el paciente estaba
acostado de lado. Dos pruebas:

1. **Barrido**: se gira el volumen varios ángulos y se mide el Dice con cada uno. Si hay un
   ángulo con el que el modelo acierta, el problema es la orientación y no la calidad de la imagen.
2. **Estimación automática**: con una primera predicción se mira hacia dónde queda el sacro
   respecto a los coxales (en un paciente bien colocado queda abajo) y se gira lo necesario. Solo
   usa "sacro contra coxales", no izquierda/derecha, que es justo lo que falla.

In [ ]:
CASO_GIRO = "012"
im, lb = datos[CASO_GIRO]["images"], datos[CASO_GIRO]["labels"]
barrido = pd.DataFrame(cd.barrido_de_giros(modelo, device, im, lb, angulos=(-135, -90, -45, 0, 45, 90, 135, 180), paso=6))
ax = barrido.plot(x="giro", y=["dice_instancias"], marker="o", figsize=(8, 3.2), legend=False)
ax.axhline(0.85, color="gray", ls="--", lw=0.8); ax.set_xlabel("giro aplicado antes del modelo (grados)"); ax.set_ylabel("Dice de instancias")
plt.show()
barrido[["giro", "dice_instancias", "dice_primario", "dice_secundario", "hueso_tomado_por_otra_region_%"]]

In [ ]:
print("ángulo del sacro respecto a los coxales (0 = abajo, lo normal):")
for c in CASOS:
    print(f"  caso {c}: predicción {cd.angulo_del_sacro(pred[c]['semantica']):7.1f}°   etiqueta real {cd.angulo_del_sacro(cd.semantic_from_labels(datos[c]['labels'])):7.1f}°")

giro = cd.estimar_giro(pred[CASO_GIRO]["semantica"])
print(f"\ngiro estimado para el caso {CASO_GIRO}: {giro:.1f}°")
im2, lb2 = cd.girar(im, lb, giro)
p2 = cd.predecir(modelo, im2, device)
m2 = cd.metricas(p2, lb2)
display(pd.DataFrame({"tal como viene": met[CASO_GIRO]["resumen"], "alineado automáticamente": m2["resumen"]}))
cd.mosaico(im2, lb2, p2, f"caso {CASO_GIRO} alineado ({giro:.0f}°)", n=3)
plt.show()

## 6. Qué opciones hay

| Situación | Qué se puede hacer | Estado |
|---|---|---|
| 34 casos guardados girados 180° | Preprocesamiento determinista: leer la cabecera y dejarlos todos en la misma orientación (`src/orientacion.py`, bandera `--canonical`) | Experimento H4, hay que reentrenar |
| Paciente acostado de lado (012) | (a) Que la red lo aprenda: aumento con rotaciones al entrenar. (b) Alinear antes de predecir: primera pasada, estimar el ángulo del sacro, girar, segunda pasada | (a) experimento H3; (b) prototipo en la sección 5b |
| Volumen de pocas filas y espaciado grueso (039) | Conservar la proporción real (`--mode pad`) y simular baja resolución al entrenar | Experimentos H2 y H3 |
| Dónde se parte el hueso (087, 078) | No es de orientación: más resolución (512) y coherencia entre cortes | Caché a 512 construido |

**Apuntes propios:**

- 
